# Reference analysis notebook

This notebook is provided as reference analysis code for the accepted paper figures. It is not intended to be a standalone reproduction package. Model weights, LoRA adapters, datasets, and intermediate hidden-state files are not included. Local paths under `data/`, `adapters/`, and `outputs/` should be adjusted to the user's environment.

The access ratio is computed from decoder-layer outputs before the final model norm, using the shared base-model unembedding subspace.


In [ ]:
# Figures: unembedding-subspace access ratio and ours-minus-baseline delta by layer.
# Required inputs: baseline and ours LoRA adapter directories with adapter_config.json and adapter_model.bin.
# The notebook builds the output-relevant subspace from the shared base model and saves access curves.

from pathlib import Path
import csv
import json
import random
from dataclasses import dataclass
from typing import Optional
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

@dataclass
class Args:
    baseline_adapter_path: Path = Path("adapters/baseline/adapter_model.bin")
    ours_adapter_path: Path = Path("adapters/ours/adapter_model.bin")
    k: int = 64
    vocab_sample: int = 20000
    seed: int = 0
    pooling: str = "last"
    max_len: int = 256
    batch_size: int = 1
    out_dir: Path = Path("outputs/unembedding_access")
    dtype: str = "bf16"
    device_map: str = "auto"
    prompts_path: Optional[Path] = None

args = Args()
args.out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
DEFAULT_PROMPTS = [
    "Solve: If x + 3 = 10, what is x? Explain briefly.",
    "A shop sells 3 apples for $2. How much do 10 apples cost? Show steps.",
    "Compute: (17 * 23) - (19 * 21).",
    "Explain why sqrt(2) is irrational in a short proof sketch.",
    "Given: All bloops are razzies. Some razzies are lazzies. Can we conclude some bloops are lazzies? Explain.",
    "Explain why the sky is blue in one paragraph.",
    "Write a short summary of the plot of Romeo and Juliet.",
    "What is the capital of France? Answer with one word.",
    "Give two reasons why regularization helps generalization in machine learning.",
    "Define covariance matrix and give one practical use case.",
]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def infer_base_model_from_adapter(adapter_path):
    cfg_path = adapter_path.parent / "adapter_config.json"
    if not cfg_path.is_file():
        raise FileNotFoundError(f"adapter_config.json not found: {cfg_path}")
    with cfg_path.open("r", encoding="utf-8") as f:
        cfg = json.load(f)
    base_model_name = cfg.get("base_model_name_or_path")
    if not base_model_name:
        raise ValueError(f"base_model_name_or_path missing in {cfg_path}")
    return base_model_name

def get_torch_dtype(dtype_str):
    return {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}[dtype_str]

def load_causal_lm(model_name, dtype, device_map=None, low_cpu_mem_usage=True):
    try:
        return AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map=device_map, low_cpu_mem_usage=low_cpu_mem_usage)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype, device_map=device_map, low_cpu_mem_usage=low_cpu_mem_usage)

def load_prompts(path):
    if path is None:
        return DEFAULT_PROMPTS
    if not path.is_file():
        raise FileNotFoundError(f"Prompts file not found: {path}")
    prompts = [line.rstrip("\n") for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not prompts:
        raise ValueError("Prompts file is empty")
    return prompts

def save_csv(path, baseline, ours, delta):
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["layer", "access_baseline", "access_ours", "delta"])
        for i in range(len(baseline)):
            writer.writerow([i + 1, float(baseline[i]), float(ours[i]), float(delta[i])])

In [ ]:
@torch.no_grad()
def build_u_from_unembedding(model, k, vocab_sample, seed):
    w = model.get_output_embeddings().weight
    vocab_size, hidden_dim = w.shape
    if k > hidden_dim:
        raise ValueError(f"k={k} exceeds hidden dimension {hidden_dim}")
    n = min(vocab_sample, vocab_size)
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)
    idx = torch.randperm(vocab_size, generator=generator, device="cpu")[:n]
    ws = w.detach().to("cpu", dtype=torch.float16)[idx]
    cov = (ws.T @ ws).float()
    _, evecs = torch.linalg.eigh(cov)
    return evecs[:, -k:].contiguous()

@torch.no_grad()
def layerwise_access_ratio(model, tokenizer, u, prompts, max_len, pooling, batch_size):
    if pooling not in {"last", "mean"}:
        raise ValueError("pooling must be 'last' or 'mean'")
    device = next(model.parameters()).device
    u = u.to(device=device, dtype=torch.float32)
    core = model.base_model.model if hasattr(model, "base_model") and hasattr(model.base_model, "model") else model
    if not (hasattr(core, "model") and hasattr(core.model, "layers")):
        raise RuntimeError("Cannot locate decoder layers at model.model.layers")
    layers = core.model.layers
    num = torch.zeros(len(layers), device=device, dtype=torch.float64)
    den = torch.zeros(len(layers), device=device, dtype=torch.float64)
    ctx = {"attn": None, "lengths": None}
    handles = []
    def make_hook(layer_idx):
        def hook_fn(module, inputs, output):
            h = output[0] if isinstance(output, (tuple, list)) else output
            h = h.float()
            attn = ctx["attn"]
            lengths = ctx["lengths"]
            b, _, d = h.shape
            if pooling == "last":
                last_pos = (lengths - 1).view(b, 1, 1).expand(b, 1, d)
                pooled = h.gather(dim=1, index=last_pos).squeeze(1)
            else:
                mask = attn.unsqueeze(-1).float()
                pooled = (h * mask).sum(dim=1) / lengths.unsqueeze(-1).float()
            coeff = pooled @ u
            num[layer_idx] += (coeff * coeff).sum(dim=1).double().sum()
            den[layer_idx] += (pooled * pooled).sum(dim=1).clamp(min=1e-12).double().sum()
            return output
        return hook_fn
    try:
        for i, layer in enumerate(layers):
            handles.append(layer.register_forward_hook(make_hook(i)))
        for i in range(0, len(prompts), batch_size):
            enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", truncation=True, max_length=max_len, padding=True)
            input_ids = enc["input_ids"].to(device)
            attn = enc["attention_mask"].to(device)
            ctx["attn"] = attn
            ctx["lengths"] = attn.long().sum(dim=1).clamp(min=1)
            model(input_ids=input_ids, attention_mask=attn, output_hidden_states=False, use_cache=False)
            if torch.cuda.is_available() and device.type == "cuda":
                torch.cuda.synchronize()
    finally:
        for handle in handles:
            handle.remove()
    return (num / den).clamp(0.0, 1.0).detach().cpu().numpy()

In [ ]:
set_seed(args.seed)
base_from_baseline = infer_base_model_from_adapter(args.baseline_adapter_path)
base_from_ours = infer_base_model_from_adapter(args.ours_adapter_path)
if base_from_baseline != base_from_ours:
    raise ValueError("Adapters must use the same base model")
base_model_name = base_from_baseline

tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_cpu = load_causal_lm(base_model_name, torch.float16, device_map=None, low_cpu_mem_usage=False)
base_cpu.eval()
u = build_u_from_unembedding(base_cpu, args.k, args.vocab_sample, args.seed)
del base_cpu

dtype = get_torch_dtype(args.dtype)
device_map = {"": 0} if args.device_map == "auto" else None if args.device_map == "cpu" else args.device_map
base_model = load_causal_lm(base_model_name, dtype, device_map=device_map, low_cpu_mem_usage=True)
base_model.eval()
peft_model = PeftModel.from_pretrained(base_model, str(args.baseline_adapter_path.parent), adapter_name="baseline", is_trainable=False)
peft_model.load_adapter(str(args.ours_adapter_path.parent), adapter_name="ours", is_trainable=False)
peft_model.eval()
prompts = load_prompts(args.prompts_path)

peft_model.set_adapter("baseline")
access_baseline = layerwise_access_ratio(peft_model, tokenizer, u, prompts, args.max_len, args.pooling, args.batch_size)
peft_model.set_adapter("ours")
access_ours = layerwise_access_ratio(peft_model, tokenizer, u, prompts, args.max_len, args.pooling, args.batch_size)
delta = access_ours - access_baseline
x = np.arange(1, len(access_baseline) + 1)

np.save(args.out_dir / "access_ratio_baseline.npy", access_baseline)
np.save(args.out_dir / "access_ratio_ours.npy", access_ours)
np.save(args.out_dir / "access_ratio_delta.npy", delta)
np.save(args.out_dir / "access_ratio.npy", np.stack([access_baseline, access_ours, delta], axis=0))
save_csv(args.out_dir / "access_ratio.csv", access_baseline, access_ours, delta)

plt.figure()
plt.plot(x, access_baseline, label="baseline", color="orange")
plt.plot(x, access_ours, label="ours", color="blue")
plt.xlabel("Layer")
plt.ylabel("Access ratio")
plt.title(f"Unembedding-subspace access (k={args.k}, pooling={args.pooling})")
plt.legend()
plt.tight_layout()
plt.savefig(args.out_dir / "access_ratio.png", dpi=220, bbox_inches="tight")
plt.show()

plt.figure()
plt.plot(x, delta, label="delta", color="blue")
plt.xlabel("Layer")
plt.ylabel("Delta access ratio")
plt.title("Delta access (ours - baseline)")
plt.tight_layout()
plt.savefig(args.out_dir / "access_ratio_delta.png", dpi=220, bbox_inches="tight")
plt.show()